In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [2]:
data_ace_24 = pd.read_csv("Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [3]:
def build_models_adaptation():
    models = {
        "linreg": LinearRegression(),
        "boost": LGBMRegressor(
            n_estimators=800,
            learning_rate=0.02,
            max_depth=3,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=42,
            verbose=-1),
        "pls": PLSRegression(n_components=35,
            scale=True,
            max_iter=2000)}
    return models

def adapt_discover_to_ace(ace_df, disc_df, all_targets, split_date="2021-01-01", adapt_model_name="linreg"):
    """
    Адаптация discover -> ace
    1. L обучается только на пересечении тренировочных данных ace и discover
    2. Тест discover адаптируется в домен данных ace
    3. Из всех данных возвращается только discover_test_adapted для дальнейшего предсказания
    """
    ace = ace_df.sort_index()
    disc = disc_df.sort_index()
    split_date = pd.Timestamp(split_date)

    ace_train = ace.loc[:split_date]
    disc_train = disc.loc[:split_date]
    disc_test = disc.loc[split_date:]   # адаптируем только тест discover

    # Пересечение индексов внутри train
    overlap_idx = ace_train.index.intersection(disc_train.index)

    ace_overlap = ace_train.loc[overlap_idx]
    disc_overlap = disc_train.loc[overlap_idx]

    feature_cols = [c for c in ace.columns if c not in all_targets]
    
    # Масштабирование
    sc_ace = StandardScaler().fit(ace_overlap[feature_cols])
    sc_disc = StandardScaler().fit(disc_overlap[feature_cols])

    X_ace = sc_ace.transform(ace_overlap[feature_cols])
    X_disc = sc_disc.transform(disc_overlap[feature_cols])

   # Выбор модели адаптации L: discover -> ace
    models = build_models_adaptation()
    
    if adapt_model_name not in models:
        raise ValueError(f"Неизвестная модель: {adapt_model_name}. Доступны: {list(models.keys())}")
    
    L = models[adapt_model_name]
    
    # Используем, так как таргет многомерный
    if adapt_model_name == "boost":
        from sklearn.multioutput import MultiOutputRegressor
        L = MultiOutputRegressor(L)

    L.fit(X_disc, X_ace)

    X_disc_test = sc_disc.transform(disc_test[feature_cols])            # Нормировка тестового набора данных discover
    X_disc_test_adapted = L.predict(X_disc_test)                        # Применение обученной модели адаптации
    X_disc_test_adapted = sc_ace.inverse_transform(X_disc_test_adapted) # Приводим новые адаптированные данные к ненормированному виду

    disc_test_adapted = pd.DataFrame(X_disc_test_adapted, index=disc_test.index, columns=feature_cols)

    # Целевые переменные возвращаются обратно неизменёнными
    for col in all_targets:
        disc_test_adapted[col] = disc_test[col]

    return disc_test_adapted, L, sc_disc, sc_ace

In [11]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("boost", LGBMRegressor(
        n_estimators=700,
        learning_rate=0.02,
        max_depth=-1, #5,
        num_leaves=30,
        subsample=0.9,
        colsample_bytree=0.8,
        early_stopping_rounds=50,
        random_state=random_state,
        verbose=-1))
    ])
    
    models['MLP'] = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
        hidden_layer_sizes=(256, 256, 128),
        activation='tanh',
        solver='adam',
        alpha=1e-4,
        learning_rate_init=1e-3,
        early_stopping=True,
        validation_fraction=0.1,
        max_iter=1500,
        random_state=random_state))
    ])
    return models
    
models = build_models()

def evaluate_M_A(ace_df, disc_test_adapted_df, split_date, all_targets, target, adaptation_method='linreg', delays='24h', results_list=None):
    """
    Модель M_A обучается на ace_train и тестируется на адаптированном discover_test
    """
    split_date = pd.Timestamp(split_date)

    ace_train = ace_df.loc[:split_date]

    feature_cols = [c for c in ace_train.columns if c not in all_targets]

    X_train = ace_train[feature_cols].values
    y_train = ace_train[target].values

    X_test = disc_test_adapted_df[feature_cols].values
    y_test = disc_test_adapted_df[target].values

    for name, model in models.items():
        if name == 'LGBM':
            # Для бустинга выделяем валидационный набор
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_train, y_train, test_size=0.1, random_state=42)

            model.fit(X_tr, y_tr,
                boost__eval_set=[(X_val, y_val)],
                boost__eval_metric='l2')
        else:
            model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        if results_list is None:
            results_list = []
            
        results_list.append({
            'Target': target,
            'Delays': delays,
            'Adaptation_method': adaptation_method,
            'Forecast_model': name,
            'RMSE': rmse,
            'R2': r2
        })
        print(f"{name}: rmse={rmse:.4f}, r2={r2:.4f}")

    return results_list

In [6]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# 1. Задание переменных для адаптации
split_date = "2021-01-01"
targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3']

In [7]:
# 2.1. Доменная адаптация с помощью моделей линейной регрессии (linreg), грудиентного бустинга (boost) и метода проекций на латентные структуры (pls)
disc_adapted_lin_24, L_lin_24, sd_lin_24, sa_lin_24 = adapt_discover_to_ace(
    data_ace_24_copy,
    data_discover_24_copy,
    targets,
    split_date,
    adapt_model_name="linreg"
)

disc_adapted_gbr_24, L_gbr_24, sd_gbr_24, sa_gbr_24 = adapt_discover_to_ace(
    data_ace_24_copy,
    data_discover_24_copy,
    targets,
    split_date,
    adapt_model_name="boost"
)

disc_adapted_pls_24, L_pls_24, sd_pls_24, sa_pls_24 = adapt_discover_to_ace(
    data_ace_24_copy,
    data_discover_24_copy,
    targets,
    split_date,
    adapt_model_name="pls"
)

In [ ]:
# # 2.2. Доменная адаптация с помощью моделей линейной регрессии (linreg), грудиентного бустинга (boost) и метода проекций на латентные структуры (pls)
# disc_adapted_lin_af, L_lin_af, sd_lin_af, sa_lin_af = adapt_discover_to_ace(
#     data_ace_af_copy,
#     data_discover_af_copy,
#     targets,
#     split_date,
#     adapt_model_name="linreg"
# )

# disc_adapted_gbr_af, L_gbr_af, sd_gbr_af, sa_gbr_af = adapt_discover_to_ace(
#     data_ace_af_copy,
#     data_discover_af_copy,
#     targets,
#     split_date,
#     adapt_model_name="boost"
# )

# disc_adapted_pls_af, L_pls_af, sd_pls_af, sa_pls_af = adapt_discover_to_ace(
#     data_ace_af_copy,
#     data_discover_af_copy,
#     targets,
#     split_date,
#     adapt_model_name="pls"
# )

In [17]:
results_list = []

print(f"\n==== Depth - 24h ====")

print(f"\n==== Adaptation - linreg ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_A(data_ace_24_copy, disc_adapted_lin_24, split_date, targets, target_col, adaptation_method='linreg', delays='24h', results_list=results_list)

print(f"\n==== Adaptation - lgbm ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_A(data_ace_24_copy, disc_adapted_gbr_24, split_date, targets, target_col, adaptation_method='lgbm', delays='24h', results_list=results_list)

print(f"\n==== Adaptation - pls ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = evaluate_M_A(data_ace_24_copy, disc_adapted_pls_24, split_date, targets, target_col, adaptation_method='pls', delays='24h', results_list=results_list)


==== Depth - 24h ====

==== Adaptation - linreg ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=3.2606, r2=0.9638
Ridge: rmse=3.2606, r2=0.9638
Lasso: rmse=3.2599, r2=0.9638
LGBM: rmse=3.2682, r2=0.9636
MLP: rmse=4.3004, r2=0.9370

==== Forecast of DST_PLUS2 ====
Linear: rmse=5.0405, r2=0.9134
Ridge: rmse=5.0404, r2=0.9134
Lasso: rmse=5.0400, r2=0.9135
LGBM: rmse=4.8492, r2=0.9199
MLP: rmse=5.7981, r2=0.8855

==== Forecast of DST_PLUS3 ====
Linear: rmse=6.4376, r2=0.8588
Ridge: rmse=6.4376, r2=0.8588
Lasso: rmse=6.4375, r2=0.8588
LGBM: rmse=6.1906, r2=0.8694
MLP: rmse=7.4791, r2=0.8094

==== Adaptation - lgbm ====

==== Forecast of DST_PLUS1 ====
Linear: rmse=3.5805, r2=0.9563
Ridge: rmse=3.5805, r2=0.9563
Lasso: rmse=3.5812, r2=0.9563
LGBM: rmse=3.5259, r2=0.9577
MLP: rmse=4.8587, r2=0.9196

==== Forecast of DST_PLUS2 ====
Linear: rmse=5.2037, r2=0.9077
Ridge: rmse=5.2037, r2=0.9077
Lasso: rmse=5.2033, r2=0.9078
LGBM: rmse=5.0276, r2=0.9139
MLP: rmse=6.2308, r2=0.8677

==== Foreca

In [ ]:
# print(f"\n==== Depth - autocorrelation function ====")

# print(f"\n==== Adaptation - linreg ====")
# for target_col in targets:
#     print(f"\n==== Forecast of {target_col.upper()} ====")
#     res = evaluate_M_A(data_ace_af_copy, disc_adapted_lin_af, split_date, targets, target_col, adaptation_method='linreg', delays='auto_func', results_list=results_list)

# print(f"\n==== Adaptation - lgbm ====")
# for target_col in targets:
#     print(f"\n==== Forecast of {target_col.upper()} ====")
#     res = evaluate_M_A(data_ace_af_copy, disc_adapted_gbr_af, split_date, targets, target_col, adaptation_method='lgbm', delays='auto_func', results_list=results_list)

# print(f"\n==== Adaptation - pls ====")
# for target_col in targets:
#     print(f"\n==== Forecast of {target_col.upper()} ====")
#     res = evaluate_M_A(data_ace_af_copy, disc_adapted_pls_af, split_date, targets, target_col, adaptation_method='pls', delays='auto_func', results_list=results_list)

In [18]:
results_df = pd.DataFrame(results_list)

In [19]:
results_df

,Target,Adaptation_Method,Forecast_Model,RMSE,R2
0,Dst_plus1,linreg,Linear,3.260586,0.963791
1,Dst_plus1,linreg,Ridge,3.260567,0.963792
2,Dst_plus1,linreg,Lasso,3.259891,0.963807
3,Dst_plus1,linreg,LGBM,3.268158,0.963623
4,Dst_plus1,linreg,MLP,4.300374,0.937016
5,Dst_plus2,linreg,Linear,5.040451,0.913442
6,Dst_plus2,linreg,Ridge,5.040440,0.913442
7,Dst_plus2,linreg,Lasso,5.039967,0.913459
8,Dst_plus2,linreg,LGBM,4.849204,0.919886
9,Dst_plus2,linreg,MLP,5.798104,0.885465


In [21]:
results_df.to_excel("results/models-adaptation-discover-to-ace.xlsx", index=False)